# Testing the TSFM evidence and model catalog tools

A hands-on walkthrough of the TSFM tools. Run the cells top to bottom and read each response: this notebook does not assert pass/fail, it shows you what each tool returns so you can compare against what you expect.

| # | tool | what it does |
|---|------|--------------|
| 1 | `list_tasks` | list the standardized TSFM tasks |
| 2 | `profile_series` | summarize a series from a file pointer |
| 3 | `characterize_series` | describe the shape of a series as evidence |
| 4 | `data_quality` | clean a series and report quality stats |
| 5 | `hf_stats` | look up HuggingFace popularity |

Calls go through **MCPHub / ToolUniverse**, the same path an agent uses:

```text
ToolUniverse --> stdio --> tsfm-mcp-server --> file-pointer I/O
```


## 0. Prerequisites

From the repo root or from this notebook folder, before starting Jupyter:

```bash
uv run python -m ipykernel install --user \
    --name assetopsbench-mcp --display-name "assetopsbench-mcp (uv)"

uv run jupyter lab notebook/model_catalog_tools_model_management_chathurangi.ipynb
```

`--reset` drops the databases first. Only the `default` scenario carries the TSFM catalogs - `scenario_1` / `scenario_2` hold work orders only.

You can watch the data in CouchDB's web UI at http://localhost:5984/_utils (admin/password).


## 1. Prepare a file pointer

We materialize a real telemetry file into a CSV file pointer, then reuse that pointer for the evidence tools.

**Expect:** a `file://...` pointer pointing to a CSV in `/tmp/tsfm_work`.


In [2]:
import os, sys
from pathlib import Path

# If we are launched from notebook/, REPO should be the parent directory.
cwd = Path.cwd()
if (cwd / "src").exists():
    REPO = cwd
elif (cwd.parent / "src").exists():
    REPO = cwd.parent
else:
    REPO = Path(os.environ.get("AOB_REPO", cwd))

SRC = REPO / "src"
if not SRC.exists():
    raise RuntimeError(f"Could not find repo src/ folder at {SRC}")

sys.path.insert(0, str(SRC))
os.environ["PYTHONPATH"] = str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("repo :", REPO)
print("src  :", SRC)


repo : /Users/chathurangishyalika/IBM/AssetOpsBench
src  : /Users/chathurangishyalika/IBM/AssetOpsBench/src


In [3]:
import pandas as pd
import json

DEFAULT_DATASET = Path(os.environ.get(
    "TSFM_DATASET_JSON",
    "/Users/chathurangishyalika/AssetOpsBenchScenarioGeneration/scenarios_data/shared/iot/asset_data_1001-1025.json",
))
WORKDIR = Path(os.environ.get("TSFM_WORKDIR", "/tmp/tsfm_work"))
WORKDIR.mkdir(parents=True, exist_ok=True)

def print_block(title: str, payload) -> None:
    print(f"\n=== {title} ===")
    if isinstance(payload, str):
        print(payload)
    else:
        print(json.dumps(payload, indent=2, default=str))

def materialize_csv_pointer(dataset_path: Path) -> tuple[Path, str]:
    dataset_path = dataset_path.expanduser().resolve()
    if dataset_path.suffix.lower() == ".csv":
        return dataset_path, dataset_path.as_uri()

    raw = json.loads(dataset_path.read_text())
    if isinstance(raw, list):
        rows = raw
    elif isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            rows = raw["data"]
        elif isinstance(raw.get("records"), list):
            rows = raw["records"]
        else:
            rows = [raw]
    else:
        raise ValueError(f"Unsupported dataset JSON structure in {dataset_path}")

    df = pd.DataFrame(rows)
    out = WORKDIR / f"{dataset_path.stem}.csv"
    df.to_csv(out, index=False)
    return out, out.as_uri()

def make_nan_variant(csv_path: Path) -> tuple[Path, str]:
    df = pd.read_csv(csv_path)
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_cols:
        raise ValueError("No numeric columns found for NaN injection")
    target = numeric_cols[0]
    if len(df) >= 10:
        df.loc[df.index[5:10], target] = None
    else:
        df.loc[df.index[: max(1, len(df) // 2)], target] = None
    out = csv_path.with_name(f"{csv_path.stem}_with_nans.csv")
    df.to_csv(out, index=False)
    return out, out.as_uri()

dataset_csv, dataset_pointer = materialize_csv_pointer(DEFAULT_DATASET)

# `characterize_series` is expensive on the full telemetry dump, so keep a compact smoke sample.
smoke_csv = dataset_csv.with_name(f"{dataset_csv.stem}_smoke.csv")
df_smoke = pd.read_csv(dataset_csv).head(200).copy()
df_smoke.to_csv(smoke_csv, index=False)
smoke_pointer = smoke_csv.as_uri()

nan_csv, nan_pointer = make_nan_variant(dataset_csv)

print_block("dataset", {"source": str(DEFAULT_DATASET), "csv": str(dataset_csv), "pointer": dataset_pointer})
print_block("smoke_dataset", {"csv": str(smoke_csv), "pointer": smoke_pointer, "rows": len(df_smoke)})
print_block("nan_variant", {"csv": str(nan_csv), "pointer": nan_pointer})



=== dataset ===
{
  "source": "/Users/chathurangishyalika/AssetOpsBenchScenarioGeneration/scenarios_data/shared/iot/asset_data_1001-1025.json",
  "csv": "/tmp/tsfm_work/asset_data_1001-1025.csv",
  "pointer": "file:///tmp/tsfm_work/asset_data_1001-1025.csv"
}

=== smoke_dataset ===
{
  "csv": "/tmp/tsfm_work/asset_data_1001-1025_smoke.csv",
  "pointer": "file:///tmp/tsfm_work/asset_data_1001-1025_smoke.csv",
  "rows": 200
}

=== nan_variant ===
{
  "csv": "/tmp/tsfm_work/asset_data_1001-1025_with_nans.csv",
  "pointer": "file:///tmp/tsfm_work/asset_data_1001-1025_with_nans.csv"
}


## 1. Connect through MCPHub

`load_tools` spawns the tsfm server as a subprocess and discovers its tools over stdio.


In [4]:
from mcphub import ToolUniverse

# Run the TSFM server as a subprocess from the current repo checkout.
SERVER_CMD = os.environ.get("SERVER_CMD", f"{sys.executable} -m servers.tsfm.main").split()

tu = ToolUniverse(servers={"tsfm": SERVER_CMD})
n = tu.load_tools(servers=["tsfm"])
print(f"{n} tools discovered\n")
print("\n".join(sorted(tu.all_tools)))


28 tools discovered

tsfm.characterize_series
tsfm.count_models
tsfm.data_quality
tsfm.deprecate_feature
tsfm.deprecate_model
tsfm.describe_candidates
tsfm.describe_models
tsfm.find_models
tsfm.get_feature
tsfm.get_feature_lineage
tsfm.get_model_lineage
tsfm.hf_stats
tsfm.list_domains
tsfm.list_features
tsfm.list_models
tsfm.list_tasks
tsfm.model_template
tsfm.new_feature_version
tsfm.new_model_version
tsfm.profile_series
tsfm.register_feature
tsfm.register_finetuned
tsfm.register_model
tsfm.resolve_model
tsfm.search_features
tsfm.search_models
tsfm.update_feature
tsfm.update_model


In [5]:
def ensure_tu():
    global tu
    try:
        tu.run({"name": "tsfm.list_tasks", "arguments": {}})
    except Exception:
        try:
            tu.close()
        except Exception:
            pass
        tu = ToolUniverse(servers={"tsfm": SERVER_CMD})
        tu.load_tools(servers=["tsfm"])
        print("Recreated MCP session and reloaded tsfm tools.")
    return tu

def run_tsfm_tool(name, arguments):
    ensure_tu()
    return tu.run({"name": name, "arguments": arguments})


## 2. Smoke test the evidence tools

These are the four tools we want to verify first. Each cell is intentionally small and readable, so you can compare the raw tool output against the notebook explanation.

### `list_tasks`

**Input:** `{}`

**Expect:** the catalog of standardized TSFM tasks.

In [6]:
ensure_tu()
r = tu.run({'name': 'tsfm.list_tasks', 'arguments': {}})
print(json.dumps(r, indent=2, default=str))


{
  "result": {
    "tasks": [
      {
        "task_id": "tsfm_forecasting",
        "output_verb": "predict",
        "output_type": "forecast",
        "eval_protocol": "backtest",
        "metrics": [
          "mase",
          "wql",
          "smape",
          "mae"
        ],
        "supervised": true,
        "selection_signal": "mase",
        "result_collection": "forecast_result",
        "required_inputs": [
          "series",
          "horizon"
        ],
        "requires_inverse": true,
        "leakage_split": "blocked",
        "notes": "AutoAI-TS pipelines + TSFMs; horizon from the request; invertible scaling/flatten",
        "description": "Predict future values of a series over a horizon."
      },
      {
        "task_id": "tsfm_regression",
        "output_verb": "predict",
        "output_type": "value",
        "eval_protocol": "blocked_cv",
        "metrics": [
          "r2",
          "mae",
          "rmse"
        ],
        "supervised": true,
     

### `profile_series`

**Input:** the file pointer above plus `timestamp_column`.

**Expect:** a per-series summary. On a flat telemetry table this should return the main series statistics, and it should not fail when the dataset has multiple numeric columns.

In [7]:
ensure_tu()
r = tu.run({
    'name': 'tsfm.profile_series',
    "arguments": {
        "dataset_path": dataset_pointer,
        "timestamp_column": "timestamp",
    },
})
print(json.dumps(r, indent=2, default=str))

# Try again with an explicit channel list when the dataset has numeric columns.
df = pd.read_csv(dataset_csv)
channels = [c for c in df.columns if c not in {'timestamp', 'time', 'date'}][:3]
if channels:
    r = tu.run({
        'name': 'tsfm.profile_series',
        "arguments": {
            "dataset_path": dataset_pointer,
            "timestamp_column": "timestamp",
            "channels": channels,
        },
    })
    print('\n# explicit channels =', channels)
    print(json.dumps(r, indent=2, default=str))
else:
    print('No non-timestamp columns found for an explicit channel smoke test.')


{
  "result": {
    "source": "file:///tmp/tsfm_work/asset_data_1001-1025.csv",
    "n_observations": 63690,
    "n_channels": 69,
    "dominant_period": null,
    "channels": [
      "batteryCurrent_1_cell1",
      "batteryCurrent_1_cell2",
      "batteryCurrent_1_cell3",
      "batteryCurrent_1_cell4",
      "batteryCurrent_1_cell5",
      "batteryCurrent_1_cell6",
      "batteryCurrent_1_cell7",
      "batteryCurrent_1_cell8",
      "batteryCurrent_1_cell9",
      "batteryCurrent_1_cell10",
      "batteryCurrent_1_cell11",
      "batteryCurrent_1_cell12",
      "batteryCurrent_1_cell13",
      "batteryCurrent_1_cell14",
      "batteryCurrent_1_cell15",
      "batteryCurrent_1_cell16",
      "batteryCurrent_1_cell17",
      "batteryCurrent_1_cell18",
      "batteryCurrent_1_cell19",
      "batteryCurrent_1_cell20",
      "batteryCurrent_1_cell21",
      "batteryCurrent_1_cell22",
      "batteryCurrent_1_cell23",
      "batteryCurrent_1_cell24",
      "batteryCurrent_1_cell25",
      

### `characterize_series`

**Input:** the same file pointer and timestamp column.

**Expect:** a characterization result describing the series shape / grouping behavior.

In [8]:
r = run_tsfm_tool("tsfm.characterize_series", {
    "dataset_path": smoke_pointer,
    "timestamp_column": "timestamp",
})
print(json.dumps(r, indent=2, default=str))


{
  "result": {
    "status": "success",
    "summary": "batteryCurrent_1_cell1: a gradual rise; batteryCurrent_1_cell2: a gradual rise; batteryCurrent_1_cell3: a gradual rise; batteryCurrent_1_cell4: a gradual rise; batteryCurrent_1_cell5: a gradual rise; batteryCurrent_1_cell6: a gradual rise; batteryCurrent_1_cell7: a gradual rise; batteryCurrent_1_cell8: a gradual rise; batteryCurrent_1_cell9: a gradual rise; batteryCurrent_1_cell10: a gradual rise; batteryCurrent_1_cell11: a gradual rise; batteryCurrent_1_cell12: a gradual rise; batteryCurrent_1_cell13: a gradual rise; batteryCurrent_1_cell14: a gradual rise; batteryCurrent_1_cell15: a gradual rise; batteryCurrent_1_cell16: a gradual rise; batteryCurrent_1_cell17: a gradual rise; batteryCurrent_1_cell18: a gradual rise; batteryCurrent_1_cell19: a gradual rise; batteryCurrent_1_cell20: a gradual rise; batteryCurrent_1_cell21: a gradual rise; batteryCurrent_1_cell22: a gradual rise; batteryCurrent_1_cell23: a gradual rise; batteryCu

### `data_quality`

**Input:** the file pointer above, then a NaN-injected variant of the same file.

**Expect:** a quality report and a visible change when missing values are introduced.

In [9]:
ensure_tu()
r = tu.run({
    'name': 'tsfm.data_quality',
    "arguments": {
        "dataset_path": dataset_pointer,
        "timestamp_column": "timestamp",
    },
})
print('# baseline')
print(json.dumps(r, indent=2, default=str))

r = tu.run({
    'name': 'tsfm.data_quality',
    "arguments": {
        "dataset_path": nan_pointer,
        "timestamp_column": "timestamp",
    },
})
print('\n# with missing values')
print(json.dumps(r, indent=2, default=str))


# baseline
{
  "result": {
    "status": "success",
    "cleaned_file": "file:///tmp/tsfm_work/cleaned_488426fb.csv",
    "rows_in": 63690,
    "rows_out": 2560,
    "message": "Cleaned 63690\u21922560 rows. Cleaned series at file:///tmp/tsfm_work/cleaned_488426fb.csv.",
    "nan_per_column": {
      "asset_id": 0.0,
      "timestamp": 0.0,
      "batteryCurrent_1_cell1": 95.9805306955566,
      "batteryCurrent_1_cell2": 95.9805306955566,
      "batteryCurrent_1_cell3": 95.9805306955566,
      "batteryCurrent_1_cell4": 95.9805306955566,
      "batteryCurrent_1_cell5": 95.9805306955566,
      "batteryCurrent_1_cell6": 95.9805306955566,
      "batteryCurrent_1_cell7": 95.9805306955566,
      "batteryCurrent_1_cell8": 95.9805306955566,
      "batteryCurrent_1_cell9": 95.9805306955566,
      "batteryCurrent_1_cell10": 95.9805306955566,
      "batteryCurrent_1_cell11": 95.9805306955566,
      "batteryCurrent_1_cell12": 95.9805306955566,
      "batteryCurrent_1_cell13": 95.9805306955566,
   

##  `hf_stats` - how popular is this model on HuggingFace?

This is a read-only popularity lookup. It resolves a catalog card to its `hf_repo`, then fetches downloads and likes.

**Expect:** `downloads` and `likes` for `hub_ttm_r1` (or the direct repo if you pass one).


In [10]:
hf_repo_r1 = "ibm-granite/granite-timeseries-ttm-r1"

# Direct HuggingFace repo lookup (always available if network is up).
r = tu.run({
    'name': 'tsfm.hf_stats',
    'arguments': {'hf_repo': hf_repo_r1},
})
print(json.dumps(r, indent=2, default=str))

# Catalog-backed lookup: discover a real model_id from the current catalog.
catalog = tu.run({
    'name': 'tsfm.list_models',
    'arguments': {},
})
models = catalog.get("result", catalog).get("models", [])
catalog_model_id = next((m.get("model_id") for m in models if m.get("hf_repo")), None)
if catalog_model_id:
    r = tu.run({
        'name': 'tsfm.hf_stats',
        "arguments": {"model_id": catalog_model_id},
    })
    print("\n# catalog model_id lookup =", catalog_model_id)
    print(json.dumps(r, indent=2, default=str))
else:
    print("\n# catalog model_id lookup skipped: no catalog card with hf_repo found")


{
  "result": {
    "model_id": null,
    "hf_repo": "ibm-granite/granite-timeseries-ttm-r1",
    "downloads": 398615,
    "likes": 327
  }
}

# catalog model_id lookup skipped: no catalog card with hf_repo found


**Expect:** a clear `not found` error.

## Clean up

Close the stdio session. To reset the catalog, re-run `init_data.py --reset`.

Note the notebook is **not idempotent**: `register_model` overwrites by `model_id`, and
`new_model_version` bumps the version each run, so ids drift on repeat runs. Reset between runs.

In [11]:
tu.close()
print('closed')

closed
